In [ ]:
import pandas as pd 
import numpy as np
# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle   

# Project imports
from Token_generation import *
from VAE_Model import *
from denoise_model import *


# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()

# Reproducibility
torch.manual_seed(0)
np.random.seed(13)
rng = np.random.default_rng()


In [ ]:
#loading the demo data
with open("files/demo_material_samples.pkl", "rb") as f:
    processing_demo, compositions_demo, properties_demo = pickle.load(f)


'''
Properties indices:
 0: 'Yield strength'
 1: 'Tensile strength'
 2: 'Vickers Hardness'
'''

'''
Composition feature indices (percentages):

  0: 'Al (aluminum) (%)'
  1: 'B (boron) (%)'
  2: 'C (carbon) (%)'
  3: 'Co (cobalt) (%)'
  4: 'Cr (chromium) (%)'
  5: 'Cu (copper) (%)'
  6: 'Fe (iron) (%)'
  7: 'Mg (magnesium) (%)'
  8: 'Mn (manganese) (%)'
  9: 'Mo (molybdenum) (%)'
 10: 'N (nitrogen) (%)'
 11: 'Nb (niobium) (%)'
 12: 'Ni (nickel) (%)'
 13: 'P (phosphorus) (%)'
 14: 'Pb (lead) (%)'
 15: 'S (sulfur) (%)'
 16: 'Se (selenium) (%)'
 17: 'Si (silicon) (%)'
 18: 'Sn (tin) (%)'
 19: 'Ta (tantalum) (%)'
 20: 'Ti (titanium) (%)'
 21: 'V (vanadium) (%)'
 22: 'W (Tungsten) (%)'
'''


In [ ]:
#loading the scaler we used to scale the properties in the training data, so that we can unscale the predictions later
with open("files/scaler_y.pkl", "rb") as f:
    scaler_y = pickle.load(f)

X = compositions_demo
Y = properties_demo

XX = X.values
YY = scaler_y.transform(Y)

In [ ]:
# Loading the tokens dictionaries back:
with open("files/token_dicts.pkl", "rb") as f:
    dicts = pickle.load(f)
CHAR_DICT = dicts["CHAR_DICT"]
ORG_DICT = dicts["ORG_DICT"]    


In [ ]:
# Conver the sentences to encoded tokens

processing = processing_demo.values 
encoded_token = vae_data_gen(processing[:,0], CHAR_DICT, max_len=127)   #max_len=126

print("Encoded tensor shape:", encoded_token.shape)
print("First sentence tokens:", word_tokenizer(processing[0,0]))
print("Encoded IDs:", encoded_token[0])

# Decode back
decoded_token = decode_words(encoded_token[:,1:], ORG_DICT)
print("Decoded sentences:", decoded_token[0])

In [ ]:
vae = vaeModel().to(device)
P_model = Pmodel().to(device)

vae.load_state_dict(torch.load('files/VAE_model.pt'))
P_model.load_state_dict(torch.load('files/P_model.pt'))

vae.eval()
P_model.eval()

In [ ]:
# VAE Tranformer inference

pad_idx = CHAR_DICT["_"]
src = torch.as_tensor(encoded_token, dtype=torch.long, device=device)         # (B, 127)
Comp = torch.as_tensor(XX, dtype=torch.float32, device=device)                # (B, 23)
prop_true_scaled = torch.as_tensor(YY, dtype=torch.float32, device=device)    # (B, 3)

src_mask = (src != pad_idx).unsqueeze(-2)  # (B, 1, T)

with torch.no_grad():
    z, mu, logvar, pred_len = vae.encoder(src, src_mask, Comp)

    # Properties prediction 
    prop_pred_scaled = P_model(mu)     # (B, 3)

    # Decode predicted processing + predicted composition distribution
    # Note: greedy_decode returns (decoded_processing_tokens, Comp_out)
    proc_pred_tokens, comp_pred = greedy_decode_reconstruction(vae, z, src_mask, use_gpu=False)


# Decode processing tokens to text
proc_pred_sentences = decode_words(proc_pred_tokens, ORG_DICT)
proc_true_sentences = decode_words(src[:, 1:], ORG_DICT)

# Unscale properties back to original units
prop_pred = scaler_y.inverse_transform(prop_pred_scaled.cpu().numpy())
prop_true = scaler_y.inverse_transform(prop_true_scaled.cpu().numpy())

print("First TRUE processing:", proc_true_sentences[0])
print("First PRED processing:", proc_pred_sentences[0])

print("First TRUE properties [YS, TS, HV]:", prop_true[0])
print("First PRED properties [YS, TS, HV]:", prop_pred[0])

print("Predicted composition distribution shape:", comp_pred.shape) 

In [ ]:
# Diffusion model Parameters
# -----------------------------

timesteps = 100
hidden_dim_denoise = 128
n_layers_denoise = 3
n_properties = 3
dim_condition = 128
embed_dim = 64


# ============================================================
# Diffusion scheduling (controls noise level at each timestep)
# ============================================================

# Linear beta schedule over diffusion timesteps
beta_schedule = linear_beta_schedule(timesteps=timesteps)

# ============================================================
# Alpha-related quantities derived from beta
# ============================================================

# α_t = 1 - β_t
alpha_schedule = 1.0 - beta_schedule

# Cumulative product of α across time: \bar{α}_t
# Represents how much original signal remains at timestep t
alpha_cumprod = torch.cumprod(alpha_schedule, axis=0)

# Previous cumulative product (used for posterior computation)
# Pads initial value with 1.0 for t=0
alpha_cumprod_prev = F.pad(alpha_cumprod[:-1], (1, 0), value=1.0)

# √(1 / α_t) — used in reverse diffusion mean prediction
sqrt_recip_alpha = torch.sqrt(1.0 / alpha_schedule)

# ============================================================
# Forward diffusion terms (q(z_t | z_0))
# ============================================================

# √\bar{α}_t — scaling factor for clean latent
sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod)

# √(1 - \bar{α}_t) — scaling factor for noise
sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alpha_cumprod)

# ============================================================
# Reverse diffusion posterior variance
# q(z_{t-1} | z_t, z_0)
# ============================================================

posterior_var = beta_schedule * (1.0 - alpha_cumprod_prev) / (1.0 - alpha_cumprod)

# ============================================================
# Denoising neural network
# Predicts noise ε(z_t, t, condition)
# ============================================================

denoise_net = DenoiseNN(
    input_dim=latent_dim,              # Dimension of latent z
    hidden_dim=hidden_dim_denoise,     # Hidden layer width
    n_layers=n_layers_denoise,         # Number of MLP layers
    n_cond=n_properties,               # Number of conditioning properties (YS, TS, HV)
    d_cond=dim_condition               # Condition embedding dimension
).to(device)



# loading the trained denoising model
checkpoint = torch.load(r"files/denoise_model.pth.tar", map_location="cpu")
denoise_net.load_state_dict(checkpoint["state_dict"])
denoise_net.eval()


In [ ]:
best_results = {}

vae.eval()
P_model.eval()
denoise_net.eval()

torch.manual_seed(217)
with torch.no_grad():

    for row_id in range(YY.shape[0]):

        # -----------------------------------
        # Condition (scaled properties)
        # -----------------------------------
        cond_scaled = torch.tensor(
            YY[row_id:row_id+1],
            device=device,
            dtype=torch.float32
        )  # (1, 3)

        # Generate 50 candidates for this alloy
        num_candidates = 50
        cond_batch = cond_scaled.repeat(num_candidates, 1) 

        # -----------------------------------
        # Diffusion sampling (batch)
        # -----------------------------------
        latent_batch = generate_samples(
            denoise_net,
            cond_batch,
            latent_dim=latent_dim,
            num_steps=timesteps,
            beta_schedule=beta_schedule,
            batch_size=num_candidates
        )[-1]  

        # -----------------------------------
        # Decode ALL samples
        # -----------------------------------
        decoded_tokens, composition_batch = greedy_decode_inference(
            vae,
            latent_batch,
            source_mask=None,
            use_gpu=False
        )  

        composition_batch = refine_composition(composition_batch)  

        decoded_sentences = decode_words(decoded_tokens, ORG_DICT) 

        # -----------------------------------
        # Add <start> token back (batch)
        # -----------------------------------
        start_col = torch.full(
            (decoded_tokens.size(0), 1),
            CHAR_DICT["<start>"],
            dtype=decoded_tokens.dtype,
            device=decoded_tokens.device
        )
        decoded_with_start = torch.cat([start_col, decoded_tokens], dim=1)  # (50, 126)

        # build mask (batch)
        src = decoded_with_start.long()
        pad_id = CHAR_DICT["_"]
        src_mask = (src != pad_id).unsqueeze(-2) 

        # -----------------------------------
        # Re-encode ALL samples at once (batch)
        # -----------------------------------
        _, mu_batch, _, _ = vae.encoder(src, src_mask, composition_batch)  # mu_batch: (50, latent_dim)

        # -----------------------------------
        # Predict properties (scaled) for ALL
        # -----------------------------------
        P_pred_scaled_batch = P_model(mu_batch) 

        # -----------------------------------
        # Inverse-transform then compute MSE (original units)
        # -----------------------------------
        P_pred_inv = scaler_y.inverse_transform(P_pred_scaled_batch.cpu().numpy()) 
        P_true_inv = scaler_y.inverse_transform(cond_scaled.cpu().numpy())

        P_pred_inv_t = torch.tensor(P_pred_inv, dtype=torch.float32)  # CPU tensor
        P_true_inv_t = torch.tensor(P_true_inv, dtype=torch.float32)  # CPU tensor

        mse_per_candidate = torch.sum((P_pred_inv_t - P_true_inv_t) ** 2, dim=1)  # (50,)
        best_idx = torch.argmin(mse_per_candidate).item()

        # -----------------------------------
        # 8) Store the sample for this alloy
        # -----------------------------------
        best_results[row_id] = {
            "P_pred": P_pred_inv[best_idx:best_idx+1],   # keep shape (1,3)
            "P_true": P_true_inv,                        # shape (1,3)
            "composition": composition_batch[best_idx:best_idx+1].cpu().numpy(),  # (1,23)
            "sequence": decoded_sentences[best_idx],     # single string
        }

# -----------------------------------
# Print results (one per alloy row)
# -----------------------------------
for row_id, result in best_results.items():
    print("\n==============================")
    print(f"Alloy row_id: {row_id}")
    print("\nProcessing sequence:\n", result["sequence"])
    print("\nComposition:\n", result["composition"])
    print("\nPredicted properties (P_pred):\n", result["P_pred"])
    print("\nTrue properties:\n", result["P_true"])
